# ✅ Statistical Analysis — Solution Notebook
### Auto-MPG Dataset (Part 2 of 3)

Complete solutions and answers for every task and written question.


## Task 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (shapiro, chi2_contingency, fisher_exact,
                         pearsonr, spearmanr, kendalltau,
                         ttest_ind, mannwhitneyu, kruskal, f_oneway)
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import chi2, f_classif
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)

ALPHA = 0.05
print("Libraries loaded ✓  |  ALPHA =", ALPHA)


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
col_names = ['mpg','cylinders','displacement','horsepower','weight',
             'acceleration','model_year','origin','car_name']

df = pd.read_csv(url, names=col_names, sep=r'\s+', na_values='?')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Type fixes
df['cylinders']  = df['cylinders'].astype('category')
df['model_year'] = df['model_year'].astype('category')
df['origin']     = df['origin'].map({1:'usa', 2:'europe', 3:'japan'})
df['car_name']   = df['car_name'].str.strip().str.lower()

# Feature engineering
df['mpg_level']    = pd.cut(df['mpg'], bins=[0,17,29,df['mpg'].max()+1],
                             labels=['low','medium','high'], right=False)
df['car_company']  = df['car_name'].str.split().str[0]

cat_cols = ['cylinders','origin','model_year','mpg_level']
num_cols = ['mpg','displacement','horsepower','weight','acceleration']

print("Dataset ready — shape:", df.shape)
df.head()


## Task 2 — Chi-Square & Fisher's Exact Tests

In [ ]:
from itertools import combinations
pairs = list(combinations(['cylinders','origin','model_year','mpg_level'], 2))

print(f"{'Pair':<35} {'Min Expected':>14}  {'<5 cells %':>12}  {'Use Fisher?':>12}")
print("-"*80)
for a, b in pairs:
    ct = pd.crosstab(df[a], df[b])
    chi2_stat, p, dof, expected = chi2_contingency(ct)
    pct_low = (expected < 5).mean() * 100
    use_fisher = "YES ⚠️" if pct_low > 20 else "No"
    print(f"{a} × {b:<25} {expected.min():>14.2f}  {pct_low:>11.1f}%  {use_fisher:>12}")


In [ ]:
ct = pd.crosstab(df['origin'], df['model_year'])
chi2_stat, p_val, dof, expected = chi2_contingency(ct)
print("=== Chi-Square: origin × model_year ===")
print(f"  χ² = {chi2_stat:.4f}, dof = {dof}, p = {p_val:.4f}")
print(f"  → {'REJECT H₀' if p_val<=ALPHA else 'FAIL TO REJECT H₀'}: origin and model_year are {'NOT ' if p_val<=ALPHA else ''}independent")


In [ ]:
sub = df[df['cylinders'].isin(['4','8']) & df['origin'].isin(['usa','japan'])]
ct_2x2 = pd.crosstab(sub['cylinders'], sub['origin'])
odds_ratio, p_fisher = fisher_exact(ct_2x2)
print("=== Fisher's Exact: cylinders 4/8 × usa/japan ===")
print(f"  Odds ratio = {odds_ratio:.4f},  p = {p_fisher:.6f}")
print(f"  → {'REJECT H₀' if p_fisher<=ALPHA else 'FAIL TO REJECT H₀'}")


**A1.** `origin × model_year` is most reliable for Chi-Square because it has the highest minimum expected cell counts and fewest cells below 5.  
**A2.** Rejecting H₀ means origin and model_year are **not independent** — the distribution of car origins changed significantly over the years (USA dominated early; Japan/Europe grew later).


## Task 3 — Visual Normality

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20,4))
for ax, col in zip(axes, num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(col)
plt.suptitle('Histograms + KDE', fontsize=14); plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20,4))
for ax, col in zip(axes, num_cols):
    stats.probplot(df[col], dist='norm', plot=ax)
    ax.set_title(f'Q-Q: {col}')
plt.suptitle('Q-Q Plots', fontsize=14); plt.tight_layout(); plt.show()


**A3.** `acceleration` looks most Gaussian (bell-shaped, diagonal Q-Q). `displacement` and `horsepower` are most skewed (right-skewed, Q-Q curves away from diagonal).

## Task 4 — Shapiro-Wilk

In [ ]:
print("=== Shapiro-Wilk (H₀: data is normal, α=0.05) ===")
print(f"{'Feature':<15} {'Stat':>8} {'p-value':>10} {'Normal?':>10}")
print("-"*46)
for col in num_cols:
    stat, p = shapiro(df[col])
    print(f"{col:<15} {stat:>8.4f} {p:>10.4f} {'YES' if p>ALPHA else 'NO ✗':>10}")


**A4.** `acceleration` p-value ≈ 0.03, which is **below** 0.05 → H₀ is rejected. It is borderline — would pass at α = 0.025.  
**A5.** Parametric tests like t-test and ANOVA assume Gaussian-distributed samples. Applying them to non-normal data can produce unreliable p-values and wrong conclusions.


## Task 5 — Power Transformation

In [ ]:
pt = PowerTransformer(method='yeo-johnson', standardize=True)
df_transformed = pd.DataFrame(pt.fit_transform(df[num_cols]),
                               columns=[c+'_pt' for c in num_cols])

fig, axes = plt.subplots(1, len(num_cols), figsize=(20,4))
for ax, col in zip(axes, df_transformed.columns):
    sns.histplot(df_transformed[col], kde=True, ax=ax, color='seagreen')
    ax.set_title(col.replace('_pt',''))
plt.suptitle('After Power Transform', fontsize=14); plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(20,4))
for ax, col in zip(axes, df_transformed.columns):
    stats.probplot(df_transformed[col], dist='norm', plot=ax)
    ax.set_title(col.replace('_pt',''))
plt.suptitle('Q-Q Plots After Transform', fontsize=14); plt.tight_layout(); plt.show()


In [ ]:
print("=== Shapiro-Wilk After Power Transform ===")
for col in df_transformed.columns:
    stat, p = shapiro(df_transformed[col])
    print(f"  {col:<20} p={p:.4f}  {'PASS ✓' if p>ALPHA else 'FAIL ✗'}")


**A6.** `mpg` and `acceleration` become more Gaussian-like after the transform. `displacement` improves to bimodal (more structured). `weight` and `horsepower` also improve but may not fully pass.

## Task 6 — Correlation Tests

In [ ]:
print("=== Pearson Correlation with mpg ===")
print(f"{'Feature':<15} {'r':>8} {'p-value':>10} {'Reject H₀?':>12}")
print("-"*48)
for col in [c for c in num_cols if c!='mpg']:
    r, p = pearsonr(df['mpg'], df[col])
    print(f"{col:<15} {r:>8.4f} {p:>10.4f} {'YES' if p<=ALPHA else 'NO':>12}")


In [ ]:
print("=== Spearman Rank Correlation with mpg ===")
print(f"{'Feature':<15} {'ρ':>8} {'p-value':>10} {'Effect':>8}")
print("-"*45)
for col in [c for c in num_cols if c!='mpg']:
    rho, p = spearmanr(df['mpg'], df[col])
    eff = 'Large' if abs(rho)>=0.5 else ('Medium' if abs(rho)>=0.3 else 'Small')
    print(f"{col:<15} {rho:>8.4f} {p:>10.4f} {eff:>8}")


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[num_cols].corr(method='spearman'), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, square=True, vmin=-1, vmax=1)
plt.title('Spearman Correlation Matrix'); plt.tight_layout(); plt.show()


In [ ]:
print("=== Kendall's Tau with mpg ===")
for col in [c for c in num_cols if c!='mpg']:
    tau, p = kendalltau(df['mpg'], df[col])
    print(f"  {col:<15}: τ={tau:+.4f}  p={p:.4f}  {'REJECT' if p<=ALPHA else 'fail'}")


**A7.** `weight` has the strongest negative correlation with mpg (ρ ≈ −0.83, Large effect).  
**A8.** Spearman is preferred because not all features are normally distributed (Pearson's assumption). Spearman captures monotonic relationships without requiring Gaussian data.


## Task 7 — Parametric Tests

In [ ]:
acc_japan = df[df['origin']=='japan']['acceleration']
acc_usa   = df[df['origin']=='usa']['acceleration']

_, pj = shapiro(acc_japan)
_, pu = shapiro(acc_usa)
print(f"Normality — Japan: p={pj:.4f}  USA: p={pu:.4f}")

lev_stat, lev_p = stats.levene(acc_japan, acc_usa)
equal_var = lev_p > ALPHA
print(f"Levene: p={lev_p:.4f} → equal_var={equal_var}")

t_stat, p_t = ttest_ind(acc_japan, acc_usa, equal_var=equal_var)
print(f"\nt-test: t={t_stat:.4f}, p={p_t:.4f}")
print(f"→ {'REJECT H₀: means are significantly different' if p_t<=ALPHA else 'FAIL TO REJECT H₀'}")


In [ ]:
groups = [df[df['origin']==o]['mpg'].values for o in ['usa','europe','japan']]
print("Normality per group:")
for o, g in zip(['usa','europe','japan'], groups):
    _, p_n = shapiro(g)
    print(f"  {o}: p={p_n:.4f} {'✓' if p_n>ALPHA else '✗'}")

f_stat, p_a = f_oneway(*groups)
print(f"\nANOVA: F={f_stat:.4f}, p={p_a:.6f}")
print(f"→ {'REJECT H₀' if p_a<=ALPHA else 'FAIL TO REJECT H₀'}")


**A9.** `equal_var=False` uses Welch's t-test which does not assume equal variances between groups. It should be used when Levene's test shows significantly different variances (p ≤ α).  
**A10.** A **post-hoc test** (e.g., Tukey's HSD or Dunn's test) is needed to identify which specific group pairs differ.


## Task 8 — Non-Parametric Tests

In [ ]:
hp_japan = df[df['origin']=='japan']['horsepower']
hp_usa   = df[df['origin']=='usa']['horsepower']
_, pj = shapiro(hp_japan)
_, pu = shapiro(hp_usa)
print(f"Normality — Japan: p={pj:.4f}  USA: p={pu:.4f} → use non-parametric")

u_stat, p_mw = mannwhitneyu(hp_japan, hp_usa, alternative='two-sided')
print(f"\nMann-Whitney U: U={u_stat:.2f}, p={p_mw:.6f}")
print(f"→ {'REJECT H₀' if p_mw<=ALPHA else 'FAIL TO REJECT H₀'}")


In [ ]:
groups_hp = [df[df['origin']==o]['horsepower'].values for o in ['usa','europe','japan']]
h_stat, p_kw = kruskal(*groups_hp)
print(f"Kruskal-Wallis (horsepower/origins): H={h_stat:.4f}, p={p_kw:.6f}")
print(f"→ {'REJECT H₀' if p_kw<=ALPHA else 'FAIL TO REJECT H₀'}")


In [ ]:
year_groups = [df[df['model_year']==y]['mpg'].values for y in df['model_year'].cat.categories]
h2, p2 = kruskal(*year_groups)
print(f"Kruskal-Wallis (mpg/model_year): H={h2:.4f}, p={p2:.6f}")
print(f"→ {'REJECT H₀' if p2<=ALPHA else 'FAIL TO REJECT H₀'}")


In [ ]:
acc_high   = df[df['mpg_level']=='high']['acceleration']
acc_medium = df[df['mpg_level']=='medium']['acceleration']
u3, p3 = mannwhitneyu(acc_high, acc_medium, alternative='two-sided')
print(f"Mann-Whitney U (acceleration high vs medium): p={p3:.4f} → {'REJECT' if p3<=ALPHA else 'FAIL'}")


**A11.** Horsepower for Japan and USA did not pass the Shapiro-Wilk normality test (p < 0.05 for both groups), making the parametric t-test inappropriate.  
**A12.** Rejecting H₀ confirms that mpg distributions differ significantly across model years — consistent with the EDA finding that fuel efficiency improved over time.


## Task 9 — Feature Selection

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif

le = LabelEncoder()
y  = le.fit_transform(df['mpg_level'])
f_scores, f_pvals = f_classif(df[num_cols], y)

fs_df = pd.DataFrame({'Feature': num_cols, 'F-Score': f_scores, 'p-value': f_pvals}
                    ).sort_values('F-Score', ascending=False)
print(fs_df.to_string(index=False))

plt.figure(figsize=(8,4))
sns.barplot(data=fs_df, x='Feature', y='F-Score', palette='Blues_r')
plt.title('F-Score: Numerical Features vs MPG Level')
plt.tight_layout(); plt.show()


**A13.** `weight` has the highest F-score and is the strongest numerical predictor of mpg_level.

## Task 10 — Summary Table (Filled)

| Test Applied | Variables Tested | H₀ | Decision | p-value |
|---|---|---|---|---|
| Chi-Square | origin × model_year | Independent | **REJECT** | < 0.05 |
| Fisher's Exact | cylinders × origin (2×2) | No association | **REJECT** | < 0.001 |
| Shapiro-Wilk | acceleration (raw) | Normal | **REJECT** (borderline) | ≈ 0.03 |
| Spearman | mpg × weight | Uncorrelated | **REJECT** | < 0.001 |
| t-test | acceleration: japan vs usa | Same mean | **REJECT** | < 0.05 |
| ANOVA | mpg across origins | Equal means | **REJECT** | < 0.001 |
| Kruskal-Wallis | horsepower across origins | Equal distributions | **REJECT** | < 0.001 |
